In [66]:
from langgraph.graph import StateGraph,START,END
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict,Annotated
from pydantic import BaseModel, Field
import operator


In [67]:
load_dotenv()

True

In [68]:
model=ChatGoogleGenerativeAI(model="gemini-3-flash-preview",temperature=0.2)

In [69]:
class Evaluation_Schema(BaseModel):
    feedback:str=Field(description="Feedback on the essay")
    score:int=Field(description="Score out of 10 for the essay",ge=0,le=10)
    


In [70]:
structured_model=model.with_structured_output(Evaluation_Schema)



In [71]:
essay="""
## Artificial Intelligence

Artificial Intelligence (AI) is one of the most transformative technologies of the modern world. It enables machines to perform tasks that normally require human intelligence, such as learning, reasoning, problem-solving, language understanding, and decision-making. AI is already being used in healthcare, education, banking, transportation, agriculture, and entertainment.

AI can improve productivity by automating repetitive tasks and helping people analyze large amounts of information quickly. In healthcare, it can assist doctors in detecting diseases and analyzing medical images. In education, AI-powered tools can provide personalized learning experiences. Businesses also use AI for customer support, fraud detection, and forecasting.

However, AI also creates challenges. Automation may replace certain jobs, while biased data can lead to unfair decisions. Privacy, cybersecurity, misinformation, and responsible use of AI are also major concerns. Therefore, AI should not be developed only for efficiency but also with strong ethical principles.

The future of AI depends on responsible innovation. Humans must ensure that AI remains a tool that strengthens society, improves lives, and supports progress rather than creating unnecessary harm.

"""

In [72]:
class UPSCstate(TypedDict):
    essay:str
    language_feedback:str
    analysis_feedback:str
    clarity_feedback:str
    overall_feedback:str
    individual_scores:Annotated[list[int], operator.add]
    avg_score:float

In [73]:
def eval_lang(state:UPSCstate)->UPSCstate:
    prompt=f"""evaluate the language in the essay as a language teacher and give feedback on basis of language quality in 50 words only and score out of 10 \n {state['essay']}"""
    structured_response=structured_model.invoke(prompt)
    return {'language_feedback':structured_response.feedback,'individual_scores':[structured_response.score]}

def eval_analysis(state:UPSCstate)->UPSCstate:
    prompt=f"""evaluate the analysis in the essay as a analysis teacher and give feedback on the basis of analysis in essay 50 words only and score out of 10 \n {state['essay']}"""
    structured_response=structured_model.invoke(prompt)
    return {'analysis_feedback':structured_response.feedback,'individual_scores':[structured_response.score]}

def eval_clarity(state:UPSCstate)->UPSCstate:
    prompt=f"""evaluate the clarity in the essay as a communication teacher and give feedback on the basis of clarity of thought in essay 50 words only and score out of 10 \n {state['essay']}"""
    structured_response=structured_model.invoke(prompt)
    return {'clarity_feedback':structured_response.feedback,'individual_scores':[structured_response.score]}

def overall_eval(state:UPSCstate)->UPSCstate:
    prompt=f"""based on the feedback and scores given by the language teacher, analysis teacher and communication teacher give overall feedback in 50 words only and score out of 10 \n {state['language_feedback']} \n {state['analysis_feedback']} \n {state['clarity_feedback']}"""
    structured_response=structured_model.invoke(prompt)
    return {'overall_feedback':structured_response.feedback,'individual_scores':[structured_response.score],'avg_score':sum(state['individual_scores'])/len(state['individual_scores'])}

In [74]:
graph=StateGraph(UPSCstate)
graph.add_node('eval_lang',eval_lang)
graph.add_node('eval_analysis',eval_analysis)
graph.add_node('eval_clarity',eval_clarity)
graph.add_node('overall_eval',overall_eval)


graph.add_edge(START,'eval_lang')
graph.add_edge(START,'eval_analysis')
graph.add_edge(START,'eval_clarity')
graph.add_edge('eval_lang','overall_eval')
graph.add_edge('eval_analysis','overall_eval')
graph.add_edge('eval_clarity','overall_eval')
graph.add_edge('overall_eval',END)

workflow=graph.compile()

In [75]:
iniitial_state={
    'essay':essay,
    'language_feedback':'',
    'analysis_feedback':'',
    'clarity_feedback':'',
    'overall_feedback':'',
    'individual_scores':[],
    'avg_score':0.0
}

In [76]:
workflow_result=workflow.invoke(iniitial_state)
print(workflow_result)

{'essay': '\n## Artificial Intelligence\n\nArtificial Intelligence (AI) is one of the most transformative technologies of the modern world. It enables machines to perform tasks that normally require human intelligence, such as learning, reasoning, problem-solving, language understanding, and decision-making. AI is already being used in healthcare, education, banking, transportation, agriculture, and entertainment.\n\nAI can improve productivity by automating repetitive tasks and helping people analyze large amounts of information quickly. In healthcare, it can assist doctors in detecting diseases and analyzing medical images. In education, AI-powered tools can provide personalized learning experiences. Businesses also use AI for customer support, fraud detection, and forecasting.\n\nHowever, AI also creates challenges. Automation may replace certain jobs, while biased data can lead to unfair decisions. Privacy, cybersecurity, misinformation, and responsible use of AI are also major con